# **Cross-validation & Grid-search**

---
## Imports

In [1]:
# cross val & grid-search
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

# modèles
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

# python lib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import time

In [2]:
# lib pour importer le dataset
from ucimlrepo import fetch_ucirepo
# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 

---
## Informations sur le dataset

In [3]:
breast_cancer_wisconsin_diagnostic.metadata

{'uci_id': 17,
 'name': 'Breast Cancer Wisconsin (Diagnostic)',
 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic',
 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv',
 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.',
 'area': 'Health and Medicine',
 'tasks': ['Classification'],
 'characteristics': ['Multivariate'],
 'num_instances': 569,
 'num_features': 30,
 'feature_types': ['Real'],
 'demographics': [],
 'target_col': ['Diagnosis'],
 'index_col': ['ID'],
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1993,
 'last_updated': 'Fri Nov 03 2023',
 'dataset_doi': '10.24432/C5DW2B',
 'creators': ['William Wolberg',
  'Olvi Mangasarian',
  'Nick Street',
  'W. Street'],
 'intro_paper': {'ID': 230,
  'type': 'NATIVE',
  'title': 'Nuclear feature extraction for breast tumor diagnosis',
  'authors': 'W. Street, W. Wolberg, O. Mangasarian',
  'venue': 'Electronic imaging',
  'yea

In [4]:
print(breast_cancer_wisconsin_diagnostic.variables) 

                  name     role         type demographic description units  \
0                   ID       ID  Categorical        None        None  None   
1            Diagnosis   Target  Categorical        None        None  None   
2              radius1  Feature   Continuous        None        None  None   
3             texture1  Feature   Continuous        None        None  None   
4           perimeter1  Feature   Continuous        None        None  None   
5                area1  Feature   Continuous        None        None  None   
6          smoothness1  Feature   Continuous        None        None  None   
7         compactness1  Feature   Continuous        None        None  None   
8           concavity1  Feature   Continuous        None        None  None   
9      concave_points1  Feature   Continuous        None        None  None   
10           symmetry1  Feature   Continuous        None        None  None   
11  fractal_dimension1  Feature   Continuous        None        

In [5]:
print(X.shape, y.shape)

(569, 30) (569, 1)


In [6]:
X.sample(4)

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
36,14.250,21.72,93.63,633.0,0.09823,0.10980,0.13190,0.05598,0.1885,0.06125,...,15.890,30.36,116.20,799.6,0.1446,0.42380,0.51860,0.14470,0.3591,0.10140
403,12.940,16.17,83.18,507.6,0.09879,0.08836,0.03296,0.02390,0.1735,0.06200,...,13.860,23.02,89.69,580.9,0.1172,0.19580,0.18100,0.08388,0.3297,0.07834
326,14.110,12.88,90.03,616.5,0.09309,0.05306,0.01765,0.02733,0.1373,0.05700,...,15.530,18.00,98.40,749.9,0.1281,0.11090,0.05307,0.05890,0.2100,0.07083
314,8.597,18.60,54.09,221.2,0.10740,0.05847,0.00000,0.00000,0.2163,0.07359,...,8.952,22.44,56.65,240.1,0.1347,0.07767,0.00000,0.00000,0.3142,0.08116


In [7]:
y.sample(4)

,Diagnosis
439,B
399,B
402,B
328,M


In [8]:
y.value_counts()
# Valeur binaire catégorielle B/M

Diagnosis
B            357
M            212
Name: count, dtype: int64

---
## Cross Validation

In [9]:
# Cross-validation demande un array de dimension 1d
y = np.ravel(y)
models = [KNeighborsClassifier(), svm.SVC(), SGDClassifier(), MLPClassifier(max_iter=300)]

for model in models:
    print(f"{model} {cross_val_score(model, X, y, cv = 5 ).mean():.2%}")



KNeighborsClassifier() 92.79%
SVC() 91.22%
SGDClassifier() 89.99%
MLPClassifier(max_iter=300) 92.44%


---
## Grid Search

In [10]:
y = np.ravel(y)
models = [KNeighborsClassifier(), svm.SVC(), SGDClassifier(), MLPClassifier()]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

In [11]:
# Définir les grilles de recherche pour chaque modèle
KNN = {'n_neighbors': [1,2,3,4,5,6,7,8,9,10], # Grille de recherche pour KNeighborsClassifier. On va tester des valeurs de voisins de 3, 5, 7 et 9
        'weights' : ['uniform', 'distance'], 
        'algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'metric' : ['cityblock', 'euclidean', 'manhattan', 'minkowski']
    }
SVM = {'kernel': ['linear', 'rbf'], # Grille de recherche pour SVC (noyau et paramètre C)
        'C': [0.1, 1, 10, 100], 
        'decision_function_shape': ['ovo', 'ovr']
    }
SGDC =  {'loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive' ], 
         'penalty': ['l2', 'l1', 'elasticnet', None],
         'fit_intercept': [False, True]
    }
MLPC = {'hidden_layer_sizes': [100, 200],
        'activation': ['identity', 'logistic', 'tanh', 'relu'],
        'solver': ['sgd', 'adam'],
        'max_iter' : [1000, 1100, 1200, 1300, 1400, 1500],
    }

param_grids = [KNN, SVM, SGDC, MLPC]

In [12]:
# Création et exécution d'un GridSearchCV pour chaque modèle dans une boucle
# On va parcourir chaque model de la liste de modele et y associer chaque valeur distinct des param_grids
for model, param_grid in zip(models, param_grids):
    start = time.time()
    gs = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)
    gs.fit(X_train, y_train)  # Entraînement du modèle sur les données d'entraînement (X, y)
    end = time.time()
    length = end - start
    # Pour chaque modele on affiche le meilleur score obtenu ainsi que les paramètres qui donnent ce score
    print(f"{length:2f} seconds!") # Afficher le temps de compute de chaque modèle
    print(f"Meilleur score pour {model.__class__.__name__}: {gs.best_score_:.3f}")  # Afficher le meilleur score
    print(f"Meilleurs paramètres pour {model.__class__.__name__}: {gs.best_params_}")  # Afficher les meilleurs paramètres
    print('=================================')

6.923867 seconds!
Meilleur score pour KNeighborsClassifier: 0.938
Meilleurs paramètres pour KNeighborsClassifier: {'algorithm': 'auto', 'metric': 'cityblock', 'n_neighbors': 9, 'weights': 'distance'}
88.307317 seconds!
Meilleur score pour SVC: 0.949
Meilleurs paramètres pour SVC: {'C': 0.1, 'decision_function_shape': 'ovo', 'kernel': 'linear'}
2.285417 seconds!
Meilleur score pour SGDClassifier: 0.927
Meilleurs paramètres pour SGDClassifier: {'fit_intercept': True, 'loss': 'modified_huber', 'penalty': 'elasticnet'}
81.571784 seconds!
Meilleur score pour MLPClassifier: 0.952
Meilleurs paramètres pour MLPClassifier: {'activation': 'logistic', 'hidden_layer_sizes': 100, 'max_iter': 1400, 'solver': 'adam'}


MLPClassifier obtient le meilleur score, mais c'est un modèle stochastique. 
Le score variera à chaque compute du modèle.

SGDClassifier obtient le moins bon score mais il est clairement le plus rapide